# **Deep Learning & Generative AI - Project**

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# **Project Overview**

### Description

The dataset for this competition consists of multiple choice questions designed to evaluate reasoning, comprehension, and answer ranking capabilities of AI systems. Each question contains a prompt along with five answer choices labeled A, B, C, D, and E.

Participants are provided with a training dataset containing questions and their corresponding correct answers. The test dataset contains similar questions without the answer column, and participants must predict the top three most probable answers for each question.

The dataset includes a variety of question styles and difficulty levels to encourage models that generalize effectively across different problem types.


### Model

The model is built strictly "from scratch" using PyTorch. No pre-trained Large Language Models (LLMs) or external word embeddings are utilized. The architecture learns purely from the provided 2000 training examples using a custom Multi-Layer Perceptron (MLP).


# **Environment Setup**

## Standard Imports

In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings("ignore")

## Workflow Reproducibility

In [3]:
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Auto-detect hardware accelerator (GPU/CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Configured Compute Device: {device}")

Configured Compute Device: cuda


# **Data Preparation**

## Loading Datasets

In [4]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

In [5]:
label_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
label_letters = ["A", "B", "C", "D", "E"]

print(f"Training instances: {len(train_df)}")
print(f"Testing instances: {len(test_df)}")

Training instances: 2000
Testing instances: 500


## Text Vectorization (TF-IDF)

Because pre-trained embeddings are restricted, we rely on a statistical Term Frequency-Inverse Document Frequency (TF-IDF) representation. The vectorizer learns a vocabulary of the top 1,000 unigrams from the corpus.


In [6]:
print("Initializing TF-IDF Vectorizer (max_features=1000)...")
tfidf = TfidfVectorizer(stop_words="english", max_features=1000)

# Build the vocabulary across all prompts and options
all_text = train_df["prompt"].tolist() + test_df["prompt"].tolist()
for l in label_letters:
    all_text.extend(train_df[l].tolist())
    all_text.extend(test_df[l].tolist())

tfidf.fit(all_text)

Initializing TF-IDF Vectorizer (max_features=1000)...


TfidfVectorizer(max_features=1000, stop_words='english')

## Tensor Construction

To format the multiple-choice structure for the neural network, the prompt is concatenated with each option individually. The resulting 5 vectors are flattened into a single high-dimensional tensor.


In [7]:
def extract_features(df):
    features = []
    print(f"Processing {len(df)} text samples...")
    for _, row in df.iterrows():
        row_features = []
        for l in label_letters:
            combined_text = str(row["prompt"]) + " " + str(row[l])
            vec = tfidf.transform([combined_text]).toarray()[0]
            row_features.extend(vec)
        features.append(row_features)
    return np.array(features, dtype=np.float32)

X_train = extract_features(train_df)
y_train = np.array([label_map[ans] for ans in train_df["answer"]], dtype=np.int64)

X_test = extract_features(test_df)
print(f"\nInput Tensor Shape: {X_train.shape}")

Processing 2000 text samples...
Processing 500 text samples...

Input Tensor Shape: (2000, 5000)


## PyTorch Dataset & DataLoader

In [8]:
class MCQDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y) if y is not None else None
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

train_dataset = MCQDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = MCQDataset(X_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# **Model Architecture**

A custom Multi-Layer Perceptron (MLP) is defined by subclassing `nn.Module`.

It utilizes two fully-connected hidden layers, ReLU non-linearities, and substantial Dropout (p=0.3) to regularize the network and prevent overfitting on the small training distribution.


## Custom MLP Definition

In [9]:
class CustomMLP(nn.Module):
    def __init__(self, input_size, num_classes=5):
        super(CustomMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        return self.network(x)

## Model Initialization

In [10]:
model = CustomMLP(input_size=X_train.shape[1]).to(device)
print("Neural Network Initialization:")
print(model)

Neural Network Initialization:
CustomMLP(
  (network): Sequential(
    (0): Linear(in_features=5000, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=5, bias=True)
  )
)


# **Training Routine**

The network is optimized using Cross-Entropy Loss (suitable for multi-class classification) and the Adam optimizer. The training loop computes gradients via backpropagation and updates the weights iteratively.


## Optimization Parameters

In [11]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 8

## Training Loop Execution

In [12]:
print("Initiating Training Loop...\n")
for epoch in range(EPOCHS):
    # Set model to training mode (enables gradient tracking)
    model.train()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Step 1: Forward pass
        y_pred = model(batch_X)
        
        # Step 2: Calculate the loss
        batch_loss = loss_fn(y_pred, batch_y)
        
        # Step 3: Zero gradients from previous iteration
        optimizer.zero_grad()
        
        # Step 4: Backpropagation (compute new gradients)
        batch_loss.backward()
        
        # Step 5: Update parameters
        optimizer.step()
        
        # Metric tracking
        running_loss += batch_loss.item() * batch_X.size(0)
        _, predicted = torch.max(y_pred, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss: {epoch_loss:.4f} | Training Accuracy: {epoch_acc:.4f}")

print("\nTraining Loop completed successfully!")

Initiating Training Loop...

Epoch [1/8] | Loss: 0.9188 | Training Accuracy: 0.7145
Epoch [2/8] | Loss: 0.0159 | Training Accuracy: 0.9975
Epoch [3/8] | Loss: 0.0036 | Training Accuracy: 0.9995
Epoch [4/8] | Loss: 0.0020 | Training Accuracy: 0.9995
Epoch [5/8] | Loss: 0.0013 | Training Accuracy: 1.0000
Epoch [6/8] | Loss: 0.0011 | Training Accuracy: 1.0000
Epoch [7/8] | Loss: 0.0009 | Training Accuracy: 1.0000
Epoch [8/8] | Loss: 0.0008 | Training Accuracy: 1.0000

Training Loop completed successfully!


# **Inference & Submission**

The trained model evaluates the hidden test set. The logits are passed through a Softmax function to obtain probability distributions, and the Top-3 highest probability classes are extracted for the final submission format.


## Test Set Predictions

In [13]:
model.eval()
sub_ids = []
sub_preds = []

print("Executing Inference on Test Set...")
with torch.no_grad():
    for i in range(len(test_df)):
        row = test_df.iloc[i]
        idx_val = row["id"] if "id" in row else i
        
        inputs = torch.tensor(X_test[i]).unsqueeze(0).to(device)
        
        # Forward pass and probability distribution
        outputs = model(inputs)
        probs = F.softmax(outputs, dim=1)
        
        # Extract Top-3 predicted options
        top3_idx = torch.topk(probs, 3, dim=-1).indices[0]
        top3_letters = [label_letters[idx] for idx in top3_idx]
        
        sub_ids.append(idx_val)
        sub_preds.append(" ".join(top3_letters))

print("Inference on Test Set executed successfully!")

Executing Inference on Test Set...
Inference on Test Set executed successfully!


## Submission File

In [14]:
submission = pd.DataFrame({"id": sub_ids, "prediction": sub_preds})
submission.to_csv("submission.csv", index=False)

print(f"\nSubmission file generated successfully. Records: {len(submission)}")
display(submission.head())


Submission file generated successfully. Records: 500


,id,prediction
0,1,A E D
1,2,B C E
2,3,B C E
3,4,E C A
4,5,C E B


# DONE